# 三规模路网算法性能对比

本 Notebook 每次运行时都会调用 C++ runner，在南山、深圳、广东三张路网的最大强连通分量内现场生成起终点，并比较 Dijkstra L4、A* L4 + MAX-8、双向 A* L4 BAE* + MAX-8。所有算法共用同一批新查询；Notebook 负责调用、汇总和科研绘图。

查询耗时以 `steady_clock` 测得的 C++ 单次调用墙钟时间为准，包含路径重建和接口缓冲区传递；路网转换、图初始化、强连通分量查询生成、地标选择和 ALT 预处理单独记录，不计入查询耗时。报告图中每个点先对同一查询的重复轮次取中位数，再汇总查询；误差线为查询间 Q1–Q3。

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / 'workspaces' / 'A3_性能').is_dir()
)
A3_DIR = REPO_ROOT / 'workspaces' / 'A3_性能'
CALLS_DIR = A3_DIR / '06_算法调用与出图'
sys.path.insert(0, str(CALLS_DIR / '前端接口'))

from performance_comparison import (
    ALGORITHMS, NETWORKS, build_frontend_response, run_comparison_suite,
)

print('数据目录：', REPO_ROOT / 'data')

In [ ]:
# 每次执行本 cell 都现场生成查询并运行算法；结果保存到本地实验输出目录。
QUERY_COUNT = 30
ROUNDS = 3
SEED = None  # None 时每次生成新种子；填写整数可复现本次点对
ALGORITHM_IDS = ('dijkstra', 'astar', 'bidirectional_astar')

from datetime import datetime
RUN_DIR = A3_DIR / '07_实验结果' / f'性能对比演示_{datetime.now():%Y%m%d_%H%M%S}'
RUN_DIR.mkdir(parents=True, exist_ok=False)
payload = run_comparison_suite(
    networks=tuple(NETWORKS), query_count=QUERY_COUNT,
    algorithms=ALGORITHM_IDS, rounds=ROUNDS, seed=SEED,
    result_dir=RUN_DIR, build_if_missing=True,
)

print('实际随机种子：', payload['networks'][0]['config']['seed'])
print('路网数：', len(payload['networks']), '；每网查询数：', QUERY_COUNT, '；轮数：', ROUNDS)
print('算法：', ', '.join(ALGORITHMS[key]['label'] for key in ALGORITHM_IDS))

In [ ]:
# 前端汇总接口：表格和图表共用这份 JSON 数据，不依赖 Notebook。
frontend_response = build_frontend_response(payload)

summary_rows = [
    {
        '路网': network['network']['label'],
        '节点数': network['network']['nodes'],
        '算法': row['label'],
        '查询数': network['config']['query_count'],
        '中位耗时 (ms)': row['median_ms'],
        'Q1 (ms)': row['q1_ms'],
        'Q3 (ms)': row['q3_ms'],
        '平均展开量': row['mean_expanded_count'],
        '相对 Dijkstra 配对提速': row['paired_speedup_vs_dijkstra'],
    }
    for network in frontend_response['networks']
    for row in network['table']
]
summary_table = pd.DataFrame(summary_rows)
display(summary_table.round(3))
print('前端响应字段：', ', '.join(frontend_response.keys()))

In [ ]:
# Draw the report figure in this Notebook; the frontend consumes chart_data.
COLORS = {
    "dijkstra": "#0072B2", "astar": "#E69F00",
    "bidirectional_astar": "#009E73",
    "bidirectional_dijkstra": "#D55E00", "ch": "#CC79A7",
}
ORDER = ("dijkstra", "astar", "bidirectional_astar", "bidirectional_dijkstra", "ch")
SHORT = {
    "dijkstra": "Dijkstra\nL4", "astar": "A*\nL4 + MAX-8",
    "bidirectional_astar": "Bi-A*\nL4 + MAX-8",
    "bidirectional_dijkstra": "Bi-Dijkstra\nB1", "ch": "CH",
}

networks = frontend_response["networks"]
selected = [key for key in ORDER if any(
    row["algorithm"] == key for network in networks for row in network["chart_data"]
)]
fig, axes = plt.subplots(1, len(networks), figsize=(max(7.5, 4.4 * len(networks)), 4.7), sharey=True)
if len(networks) == 1:
    axes = [axes]
for ax, network in zip(axes, networks):
    rows = {row["algorithm"]: row for row in network["chart_data"]}
    for x, key in enumerate(selected):
        if key not in rows:
            continue
        row = rows[key]
        values = np.asarray(row["per_query_median_ms"], dtype=float)
        jitter = (np.arange(len(values)) / max(len(values) - 1, 1) - 0.5) * 0.24
        ax.scatter(x + jitter, values, s=15, color=COLORS[key], alpha=0.42, linewidths=0)
        ax.errorbar(x, row["median_ms"],
                    yerr=[[row["median_ms"] - row["q1_ms"]], [row["q3_ms"] - row["median_ms"]]],
                    fmt="D", markersize=5, markerfacecolor="white",
                    markeredgecolor=COLORS[key], ecolor=COLORS[key], capsize=3)
        ax.annotate(f"{row['median_ms']:.2g}", (x, row["median_ms"]), xytext=(0, 7),
                    textcoords="offset points", ha="center", fontsize=7.5)
    graph = network["network"]
    ax.set_title(f"{graph['label']} | {graph['nodes']:,} nodes, {graph['directed_edges']:,} directed edges")
    ax.set_xticks(range(len(selected)), [SHORT[key] for key in selected])
    ax.set_yscale("log")
    ax.set_ylim(0.008, 300)
    ax.grid(axis="y", which="major", color="#D9DEE3", linewidth=0.75)
    ax.grid(axis="y", which="minor", color="#EDF0F2", linewidth=0.5)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_axisbelow(True)
axes[0].set_ylabel("Query time (ms, log scale)")
fig.suptitle("Shortest-path query latency across the three network scales", y=0.98, fontsize=12)
fig.text(0.5, 0.025, f"Each dot is one OD pair; diamond = median of {QUERY_COUNT} pairs, whiskers = Q1-Q3. " +
         "Algorithms share query pairs; graph setup and heuristic preprocessing are excluded.",
         ha="center", fontsize=8, color="#444444")
fig.tight_layout(rect=(0.02, 0.10, 0.995, 0.92), w_pad=1.8)
plt.show()
